# Statistical Significance Test (Wilcoxon Signed-Rank)
This notebook performs pairwise significance tests between our model and each baseline,
with Bonferroni correction for multiple comparisons.

- **Null hypothesis H₀**: The two paired distributions are identical.
- **Significance level**: $\alpha = 0.05$ (family-wise).
- **Correction**: Bonferroni-adjusted $\alpha_{\text{adj}} = 0.05 / m$, where $m$ is the number of pairwise comparisons.



In [2]:
from scipy import stats
import json
import numpy as np
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────
# H₀: The two paired distributions are identical.
# α  = 0.05 (family-wise), Bonferroni-corrected per comparison.

EVALUATE_RESULTS_PATH = Path("./outputs/evaluate_result.jsonl")
TARGET_MODEL_NAME = "Ours(DPO)"
ALPHA_FAMILY = 0.05

# Baselines to compare against
BASELINE_NAMES = [
    "BaselineA",
    "BaselineB",
    "BaselineC",
]

# ── Load data ──────────────────────────────────────────────────────
# The file is a single JSON object keyed by model name.
with open(EVALUATE_RESULTS_PATH, "r", encoding="utf-8") as f:
    evaluate_result = json.load(f)

print(f"Models in file: {list(evaluate_result.keys())}\n")

# Check that all required models exist
missing = []
if TARGET_MODEL_NAME not in evaluate_result:
    missing.append(TARGET_MODEL_NAME)
for bl in BASELINE_NAMES:
    if bl not in evaluate_result:
        missing.append(bl)
if missing:
    print(f"WARNING: missing models — {missing}")
    BASELINE_NAMES = [bl for bl in BASELINE_NAMES if bl not in missing]

# ── Helper ─────────────────────────────────────────────────────────

def wilcoxon_paired(name, list1, list2, alpha_adj):
    """Run Wilcoxon signed-rank test (two-sided) and print results."""
    list1 = np.asarray(list1, dtype=float)
    list2 = np.asarray(list2, dtype=float)

    stat, p_value = stats.wilcoxon(list1, list2, alternative='two-sided')

    # Compute Z and effect size (rank-biserial correlation)
    d = list1 - list2
    nonzero = d != 0
    d = d[nonzero]
    n = len(d)

    if n == 0:
        Z = 0.0
        r = 0.0
    else:
        ranks = stats.rankdata(np.abs(d))
        W_plus = np.sum(ranks[d > 0])
        mu_W = n * (n + 1) / 4
        sigma_W = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
        Z = (W_plus - mu_W + 0.5 * np.sign(mu_W - W_plus)) / sigma_W
        r = Z / np.sqrt(n)

    significant = p_value < alpha_adj
    return {
        "stat": stat,
        "p_value": p_value,
        "Z": Z,
        "r": r,
        "n": len(list1),
        "n_nonzero": n,
        "significant": significant,
        "alpha_adj": alpha_adj,
    }

# ── Pairwise comparisons ───────────────────────────────────────────
# Two metrics: Semantic Score, Valid Style Score
# Bonferroni: m = len(BASELINE_NAMES) × 2 (one per metric)
N_COMPARISONS = len(BASELINE_NAMES) * 2
alpha_adj = ALPHA_FAMILY / N_COMPARISONS

print("=" * 72)
print("Wilcoxon Signed-Rank — Pairwise Tests")
print(f"H₀  : distributions are identical")
print(f"α   = {ALPHA_FAMILY} (family-wise)")
print(f"α′  = {alpha_adj:.6f} (Bonferroni-adjusted, m = {N_COMPARISONS})")
print(f"H₁  : distributions differ (two-sided)")
print("=" * 72)

summary = []

for bl_name in BASELINE_NAMES:
    for metric, metric_key in [("Semantic Score", "semantic"),
                                ("Valid Style Score", "valid_style")]:
        target_scores = evaluate_result[TARGET_MODEL_NAME][metric_key]
        baseline_scores = evaluate_result[bl_name][metric_key]
        res = wilcoxon_paired(
            f"{TARGET_MODEL_NAME} vs {bl_name} — {metric}",
            target_scores, baseline_scores,
            alpha_adj,
        )
        res["baseline"] = bl_name
        res["metric"] = metric
        summary.append(res)

        sig_str = "✓ SIGNIFICANT" if res["significant"] else "✗ NS"
        print(f"\n{res['baseline']} — {res['metric']}")
        print(f"  N = {res['n']} pairs ({res['n_nonzero']} non-zero)")
        print(f"  Z = {res['Z']:.4f}, p = {res['p_value']:.6f}")
        print(f"  r = {res['r']:.4f}  (rank-biserial)")
        print(f"  p < α′ ({res['alpha_adj']:.6f})?  →  {sig_str}")

print("\n" + "=" * 72)
print("SUMMARY TABLE")
print("=" * 72)
print(f"{'Baseline':<14} {'Metric':<20} {'Z':>8} {'p':>10} {'r':>8} {'Sig?':>6}")
print("-" * 72)
for s in summary:
    bl = s["baseline"]
    m = s["metric"]
    sig = "✓" if s["significant"] else ""
    print(f"{bl:<14} {m:<20} {s['Z']:>8.4f} {s['p_value']:>10.6f} {s['r']:>8.4f} {sig:>6}")


Models in file: ['Ours(SFT)', 'Ours(DPO)', 'BaselineA', 'BaselineB', 'BaselineC', 'DPO(wo/CoT_Shared)']

Wilcoxon Signed-Rank — Pairwise Tests
H₀  : distributions are identical
α   = 0.05 (family-wise)
α′  = 0.008333 (Bonferroni-adjusted, m = 6)
H₁  : distributions differ (two-sided)

BaselineA — Semantic Score
  N = 1200 pairs (1200 non-zero)
  Z = 18.3893, p = 0.000000
  r = 0.5309  (rank-biserial)
  p < α′ (0.008333)?  →  ✓ SIGNIFICANT

BaselineA — Valid Style Score
  N = 1200 pairs (1200 non-zero)
  Z = 15.2628, p = 0.000000
  r = 0.4406  (rank-biserial)
  p < α′ (0.008333)?  →  ✓ SIGNIFICANT

BaselineB — Semantic Score
  N = 1200 pairs (1195 non-zero)
  Z = 0.1877, p = 0.851095
  r = 0.0054  (rank-biserial)
  p < α′ (0.008333)?  →  ✗ NS

BaselineB — Valid Style Score
  N = 1200 pairs (1195 non-zero)
  Z = 10.5275, p = 0.000000
  r = 0.3045  (rank-biserial)
  p < α′ (0.008333)?  →  ✓ SIGNIFICANT

BaselineC — Semantic Score
  N = 1200 pairs (1200 non-zero)
  Z = 29.2769, p = 0.00000